In [ ]:
import numpy as np
import h5py

# =========================
# Parámetros
# =========================
NUM_FREQ = 672
NUM_ANTS = 32
BLOCK_SIZE = 2
NUM_RECEIVERS = NUM_ANTS // BLOCK_SIZE
NUM_BLOCKS = NUM_RECEIVERS * (NUM_RECEIVERS + 1) // 2   # 136
SAMPLES_PER_FRAME = 14976

GOOD_ANTS = set(range(8, 32))

GPU_PATH = "/home/juan_pablo/kotekan/frames_host_corr/frame_1.bin"
#H5_PATH = "/hdd/32_test/260306T185135Z_CHARTS_hdf5/baseband_virtual.h5"
#H5_PATH= "/hdd/32_test/260310T134521Z_CHARTS_hdf5/baseband_virtual.h5"
#H5_PATH = "/hdd/32_test/260311T161858Z_CHARTS_hdf5/baseband_virtual.h5"
#H5_PATH = "/hdd/32_test/260311T191802Z_CHARTS_hdf5/baseband_virtual.h5"
H5_PATH = "/hdd/32_test/260319T154442Z_CHARTS_hdf5/baseband_virtual.h5"
FRAME = 0

# =========================
# Utilidades CPU
# =========================
def unpack_4bit_complex(u8: np.ndarray) -> np.ndarray:
    real = (u8 >> 4).astype(np.int8)
    imag = (u8 & 0x0F).astype(np.int8)
    real[real >= 8] -= 16
    imag[imag >= 8] -= 16
    return real.astype(np.float32) + 1j * imag.astype(np.float32)

def load_cpu_chains(h5_path):
    with h5py.File(h5_path, "r") as f:
        packed = f["baseband"][:, :, :]   # esperado: (ant, freq, time)
    chains = unpack_4bit_complex(packed)
    return chains

def build_cpu_pair_dict(chains, frame=0, samples_per_frame=SAMPLES_PER_FRAME):
    t0 = frame * samples_per_frame
    t1 = (frame + 1) * samples_per_frame

    cpu_dict = {}
    for a in range(NUM_ANTS):
        xa = chains[a][:, t0:t1]   # (freq, time)
        for b in range(NUM_ANTS):
            xb = chains[b][:, t0:t1]
            vis = np.sum(xa * np.conj(xb), axis=1)   # (freq,)
            cpu_dict[(a, b)] = vis
    return cpu_dict

# =========================
# Utilidades GPU
# =========================
def load_gpu_frame_raw(path):
    raw = np.fromfile(path, dtype=np.int32)
    #raw = raw.astype(np.float32)  # para evitar overflow al convertir a complejo
    expected = NUM_FREQ * NUM_BLOCKS * 4 * 2
    assert raw.size == expected, f"Tamaño inesperado: {raw.size}, esperado {expected}"
    corr = raw.reshape(NUM_FREQ, NUM_BLOCKS, 4, 2)
    V = corr[..., 0] + 1j * corr[..., 1]   # (freq, block, k)
    return V



def build_receiver_groups(num_elements=NUM_ANTS, reverse_groups=False, swap_inside=False):
    """
    Devuelve una lista de grupos de 2 elementos.
    normal: [(0,1), (2,3), ..., (30,31)]
    reverse_groups: invierte numeración global -> [(31,30), (29,28), ...] antes de swap interno final
    swap_inside: intercambia orden dentro de cada grupo
    """
    groups = []
    for r in range(num_elements // 2):
        a0 = 2 * r
        a1 = 2 * r + 1
        groups.append((a0, a1))

    if reverse_groups:
        groups = [(num_elements - 1 - a0, num_elements - 1 - a1) for (a0, a1) in groups]

    if swap_inside:
        groups = [(a1, a0) for (a0, a1) in groups]

    return groups
def build_block_labels_lower(num_receivers=NUM_RECEIVERS):
    """
    Triángulo inferior:
    (0,0), (1,0), (1,1), (2,0), (2,1), (2,2), ...
    """
    block_labels = []
    for ry in range(num_receivers):
        for rx in range(ry + 1):
            block_labels.append((ry, rx))
    assert len(block_labels) == num_receivers * (num_receivers + 1) // 2
    return block_labels

def gpu_frame_to_pair_dict(Vgpu, reverse_groups=False, swap_inside=False):
    """
    Interpreta la salida GPU como:
      block -> (ry, rx) en triángulo inferior
      k = 2*py + px
      par escalar -> (a, b) con:
          a = groups[ry][py]
          b = groups[rx][px]
    """
    block_labels = build_block_labels_lower(NUM_RECEIVERS)
    groups = build_receiver_groups(NUM_ANTS, reverse_groups=reverse_groups, swap_inside=swap_inside)

    gpu_dict = {}

    for block, (ry, rx) in enumerate(block_labels):
        gy = groups[ry]
        gx = groups[rx]

        # k=0..3
        # 0 -> (py=0, px=0)
        # 1 -> (py=0, px=1)
        # 2 -> (py=1, px=0)
        # 3 -> (py=1, px=1)
        gpu_dict[(gy[0], gx[0])] = Vgpu[:, block, 0]
        gpu_dict[(gy[0], gx[1])] = Vgpu[:, block, 1]
        gpu_dict[(gy[1], gx[0])] = Vgpu[:, block, 2]
        gpu_dict[(gy[1], gx[1])] = Vgpu[:, block, 3]

    return gpu_dict

# =========================
# Métricas
# =========================
def phase_metrics(g, c, amp_percentile=70, min_points=20):
    g = np.asarray(g)
    c = np.asarray(c)

    ag = np.abs(g)
    ac = np.abs(c)

    thr_g = np.percentile(ag, amp_percentile)
    thr_c = np.percentile(ac, amp_percentile)

    mask = (ag > thr_g) & (ac > thr_c)

    if np.sum(mask) < min_points:
        return None

    gm = g[mask]
    cm = c[mask]

    phi0 = np.angle(np.sum(gm * np.conj(cm)))
    gm_al = gm * np.exp(-1j * phi0)

    dphi = np.angle(gm_al * np.conj(cm))

    rms_deg = np.sqrt(np.mean(dphi**2)) * 180 / np.pi
    coh = np.abs(np.mean(np.exp(1j * dphi)))

    return {
        "coh": coh,
        "rms_deg": rms_deg,
        "phi0_deg": phi0 * 180 / np.pi,
        "npts": int(np.sum(mask)),
    }


In [ ]:
# CPU
chains = load_cpu_chains(H5_PATH)
x_h5 = np.transpose(chains, (1, 0, 2)) # [f,e,t]
cpu_dict = build_cpu_pair_dict(chains, frame=0)

In [ ]:
# GPU
print("GPU_PATH:", GPU_PATH)
# Matrix 5 frames


Vgpu = load_gpu_frame_raw(GPU_PATH)
gpu_dict = gpu_frame_to_pair_dict(Vgpu, reverse_groups=True, swap_inside=False)

In [ ]:
def normalize_all_pairs_to_ref(pair_dict, ref_key, eps=1e-12):
    norm_dict = {}

    for (i, j), vij in pair_dict.items():
        if ref_key not in pair_dict:
            continue

        vref = np.asarray(pair_dict[ref_key], dtype=np.complex64)
        vij = np.asarray(vij, dtype=np.complex64)

        bad = (~np.isfinite(vref)) | (np.abs(vref) < eps)

        out = np.full_like(vij, np.nan + 1j * np.nan)
        out[~bad] = vij[~bad] / vref[~bad]

        norm_dict[(i, j)] = out

    return norm_dict

In [ ]:
cpu_dict_norm = normalize_all_pairs_to_ref(cpu_dict, ref_key=(19,25))
gpu_dict_norm = normalize_all_pairs_to_ref(gpu_dict, ref_key=(19,25))

cpu_dict_norm = cpu_dict
gpu_dict_norm = gpu_dict

In [ ]:
labels = sorted(set(cpu_dict_norm.keys()) & set(gpu_dict_norm.keys()))
labels = [lab for lab in labels if lab[0] in range(8,32) and lab[1] in range(8,32) and lab[0] != lab[1]]

gpu_list = [gpu_dict_norm[lab] for lab in labels]
cpu_list = [cpu_dict_norm[lab] for lab in labels]

## Diff

In [ ]:
def phase_compare(g, c, amp_percentile=70, min_points=20):
    g = np.asarray(g)
    c = np.asarray(c)

    finite = (
        np.isfinite(g.real) & np.isfinite(g.imag) &
        np.isfinite(c.real) & np.isfinite(c.imag)
    )

    if np.sum(finite) < min_points:
        return None

    g = g[finite]
    c = c[finite]

    ag = np.abs(g)
    ac = np.abs(c)

    thr_g = np.percentile(ag, amp_percentile)
    thr_c = np.percentile(ac, amp_percentile)

    mask = (ag > thr_g) & (ac > thr_c)
    if np.sum(mask) < min_points:
        return None

    gm = g[mask]
    cm = c[mask]

    phi0 = np.angle(np.sum(gm * np.conj(cm)))
    gm_al = gm * np.exp(-1j * phi0)
    dphi = np.angle(gm_al * np.conj(cm))

    return {
        "phi0_deg": phi0 * 180 / np.pi,
        "rms_deg": np.sqrt(np.mean(dphi**2)) * 180 / np.pi,
        "coh": np.abs(np.mean(np.exp(1j * dphi))),
        "mask": mask,
    }

In [ ]:
results = []

for lab, g, c in zip(labels, gpu_list, cpu_list):
    met = phase_compare(g, c)

    if met is None:
        continue

    results.append((lab, met["coh"], met["rms_deg"], met["phi0_deg"]))

results_sorted = sorted(results, key=lambda x: (-x[1], x[2]))

for r in results_sorted[:20]:
    print(r)

In [ ]:
import matplotlib.pyplot as plt

frequencies = np.linspace(300, 501.6, 672, endpoint=False)  # Frequency axis

def plot_pair(label, gpu_dict, cpu_dict, amp_percentile=70, wrap_phase=True):
    g = gpu_dict[label]
    c = cpu_dict[label]

    met = phase_compare(g, c, amp_percentile=amp_percentile)
    if met is None:
        print("Muy pocos bins válidos")
        return

    ag = np.abs(g)
    ac = np.abs(c)

    thr_g = np.percentile(ag, amp_percentile)
    thr_c = np.percentile(ac, amp_percentile)
    mask = (ag > thr_g) & (ac > thr_c)

    phi0 = np.angle(np.sum(g[mask] * np.conj(c[mask])))
    g_al = g * np.exp(-1j * phi0)

    phase_g = np.unwrap(np.angle(g_al)) if wrap_phase else np.angle(g_al)
    phase_c = np.unwrap(np.angle(c)) if wrap_phase else np.angle(c)
    dphi = np.angle(g_al[mask] * np.conj(c[mask]))

    fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

    axs[0].plot(frequencies, phase_c, '.', ms=3, label='CPU')
    axs[0].plot(frequencies, phase_g, '.', ms=3, label='GPU alineada')
    axs[0].set_ylabel("Phase [rad]")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)

    axs[1].plot(frequencies[mask], dphi, '.', ms=3)
    axs[1].axhline(0, ls='--')
    axs[1].set_ylabel("Δphase [rad]")
    axs[1].grid(True, alpha=0.3)

    axs[2].plot(frequencies, np.abs(c), '.', ms=3, label='|CPU|')
    axs[2].plot(frequencies, np.abs(g), '.', ms=3, label='|GPU|')
    axs[2].set_ylabel("Amplitude")
    axs[2].set_xlabel("Frequency")
    axs[2].legend()
    axs[2].grid(True, alpha=0.3)

    fig.suptitle(
        f"{label} | coh={met['coh']:.3f}, RMS={met['rms_deg']:.1f}°, phi0={met['phi0_deg']:.1f}°"
    )
    plt.tight_layout()
    plt.show()

# Plot first 5 pairs
for lab, _, _, _ in results_sorted[:5]:
    plot_pair(lab, gpu_dict, cpu_dict, wrap_phase=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

selected_antennas = [8, 9, 10,
                     16, 17, 18,
                     24,26]

frequencies = np.arange(NUM_FREQ)

# construir lista de pares únicos
pairs = []
for idx_i, i in enumerate(selected_antennas):
    for j in selected_antennas[idx_i + 1:]:
        pair = (i, j)
        if pair in cpu_dict:
            pairs.append(pair)
        elif (j, i) in cpu_dict:   # por si están guardadas al revés
            pairs.append((j, i))

n_pairs = len(pairs)
n_cols = 5
n_rows = int(np.ceil(n_pairs / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes = np.atleast_1d(axes).flatten()

print("Calculating correlations and plotting all pairs in one figure...")

for plot_idx, pair in enumerate(pairs):
    i, j = pair
    corr_spectrum = cpu_dict[pair]
    phase = np.angle(corr_spectrum)

    ax = axes[plot_idx]
    ax.scatter(frequencies, phase, s=3)
    ax.set_title(f"Ant {i} vs Ant {j}", fontsize=10)

    if plot_idx >= (n_rows - 1) * n_cols:
        ax.set_xlabel("Frequency [MHz]", fontsize=8)

    if plot_idx % n_cols == 0:
        ax.set_ylabel("Phase [rad]", fontsize=8)

    ax.grid(True, alpha=0.3)

# apagar ejes sobrantes
for k in range(n_pairs, len(axes)):
    axes[k].axis("off")

fig.suptitle("Phase spectra for all selected antenna correlations", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#i = np.arange(8, 32)   # antenas fijas

for i in range(8, 32):
    n_ant = 32

    # eje de frecuencias
    frequencies = np.arange(NUM_FREQ)   # o el eje real si ya lo tienes

    # columnas y filas
    n_cols = 5
    n_pairs = len(range(8, n_ant))
    n_rows = int(np.ceil(n_pairs / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
    axes_flat = axes.flatten()

    print("Calculating correlations and plotting...")

    plot_idx = 0
    for j in range(8, n_ant):
        pair = (i, j)

        if pair not in cpu_dict:
            continue

        corr_spectrum = cpu_dict[pair]
        phase = np.angle(corr_spectrum)

        ax = axes_flat[plot_idx]
        ax.scatter(frequencies, phase, s=3)
        ax.set_title(f"Ant {i} vs Ant {j}", fontsize=10)

        if plot_idx >= n_pairs - n_cols:
            ax.set_xlabel("Frequency [MHz]", fontsize=8)

        if plot_idx % n_cols == 0:
            ax.set_ylabel("Phase [rad]", fontsize=8)

        ax.grid(True, alpha=0.3)

        plot_idx += 1

    # ocultar ejes vacíos
    for k in range(plot_idx, len(axes_flat)):
        axes_flat[k].axis("off")

    plt.tight_layout()
    fig.suptitle(f"Phase spectra for antenna i={i} vs antennas 8..{n_ant-1} on CPU", fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.98])  # deja espacio para el título global
    plt.show()

In [ ]:
frequencies = np.linspace(300, 501.6, 672, endpoint=False)
n_ant = 32
n_cols = 5
n_rows = int(np.ceil((n_ant - 8) / n_cols))
i = 8
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes_flat = axes.flatten()

print("Calculating GPU correlations and plotting...")

plot_idx = 0
for j in range(8, n_ant):
    pair = (i, j)

    if pair in gpu_dict_norm:
        corr_spectrum = gpu_dict_norm[pair]
    elif (j, i) in gpu_dict_norm:
        corr_spectrum = np.conj(gpu_dict_norm[(j, i)])   # <-- corrección clave
    else:
        continue

    phase = np.angle(corr_spectrum)

    ax = axes_flat[plot_idx]
    ax.scatter(frequencies, phase, s=3)
    ax.set_title(f"Ant {i} vs Ant {j}", fontsize=10)
    ax.grid(True, alpha=0.3)

    if plot_idx >= (n_ant - 8) - n_cols:
        ax.set_xlabel("Frequency [MHz]", fontsize=7)

    if plot_idx % n_cols == 0:
        ax.set_ylabel("Phase [rad]", fontsize=8)

    plot_idx += 1

for k in range(plot_idx, len(axes_flat)):
    axes_flat[k].axis("off")

plt.tight_layout()
fig.suptitle(f"Phase spectra for antenna i={i} vs antennas 8..{n_ant-1} on GPU", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes_flat = axes.flatten()

print("Calculating GPU lag correlations and plotting...")

plot_idx = 0
for j in range(8, n_ant):
    pair = (i, j)

    if pair in gpu_dict_norm:
        corr_spectrum = gpu_dict_norm[pair]
    elif (j, i) in gpu_dict_norm:
        corr_spectrum = gpu_dict_norm[(j, i)]
    else:
        continue

    corr_spectrum = np.nan_to_num(corr_spectrum)

    lag_spectrum = np.fft.fftshift(np.fft.ifft(corr_spectrum))
    lags = np.fft.fftshift(np.fft.fftfreq(len(corr_spectrum), d=DELTA_FREQ))
    amplitude = np.abs(lag_spectrum)

    ax = axes_flat[plot_idx]
    ax.plot(lags, amplitude)
    ax.set_title(f"Ant {i} vs Ant {j}", fontsize=10)
    ax.grid(True, alpha=0.3)

    if plot_idx >= (n_ant - 8) - n_cols:
        ax.set_xlabel("Lag", fontsize=7)

    if plot_idx % n_cols == 0:
        ax.set_ylabel("Amplitude", fontsize=8)

    plot_idx += 1

for k in range(plot_idx, len(axes_flat)):
    axes_flat[k].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
frequencies = np.linspace(300, 501.6, 672, endpoint=False)
n_ant = 32
n_cols = 5
i=16
n_rows = int(np.ceil((n_ant - 8) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
axes_flat = axes.flatten()

print("Calculating phase difference CPU - GPU and plotting...")

plot_idx = 0
for j in range(8, n_ant):
    pair = (i, j)

    # CPU
    if pair not in cpu_dict_norm:
        continue
    corr_cpu = cpu_dict_norm[pair]

    # GPU, corrigiendo orden si viene como (j,i)
    if pair in gpu_dict_norm:
        corr_gpu = gpu_dict_norm[pair]
    elif (j, i) in gpu_dict_norm:
        corr_gpu = np.conj(gpu_dict_norm[(j, i)])
    else:
        continue

    phase_cpu = np.angle(corr_cpu)
    phase_gpu = np.angle(corr_gpu)

    # diferencia de fase envuelta en [-pi, pi]
    phase_diff = np.angle(np.exp(1j * (phase_cpu - phase_gpu)))

    ax = axes_flat[plot_idx]
    ax.scatter(frequencies, phase_diff, s=3)
    ax.set_title(f"Ant {i} vs Ant {j}", fontsize=10)
    ax.grid(True, alpha=0.3)

    if plot_idx >= (n_ant - 8) - n_cols:
        ax.set_xlabel("Frequency [MHz]", fontsize=7)

    if plot_idx % n_cols == 0:
        ax.set_ylabel("Phase diff [rad]", fontsize=8)

    plot_idx += 1

for k in range(plot_idx, len(axes_flat)):
    axes_flat[k].axis("off")

plt.tight_layout()
fig.suptitle(f"Phase difference (CPU - GPU) for antenna i={i} vs antennas 8..{n_ant-1}", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("../gpu_benchmark_1min.csv")

def clean_column(col):
    return (
        col.astype(str)
        .str.replace('%','', regex=False)
        .str.replace('MiB','', regex=False)
        .str.replace('W','', regex=False)
        .str.replace('MHz','', regex=False)
        .str.strip()
        .astype(float)
    )

df["gpu_util"] = clean_column(df[" utilization.gpu [%]"])
df["mem_util"] = clean_column(df[" utilization.memory [%]"])
df["mem_used"] = clean_column(df[" memory.used [MiB]"])
df["temp"] = clean_column(df[" temperature.gpu"])
df["power"] = clean_column(df[" power.draw [W]"])
df["clock_sm"] = clean_column(df[" clocks.current.sm [MHz]"])

# Eliminar las primeras 25 filas
df = df.iloc[25:].reset_index(drop=True)

plt.figure(figsize=(10,6))

plt.plot(df["gpu_util"], label="GPU utilization (%)")
plt.plot(df["temp"], label="Temperature (C)")
plt.plot(df["power"], label="Power (W)")
plt.plot(df["mem_util"], label="Memory utilization (%)")

plt.title("GPU Metrics Over Time During Correlation")
plt.xlabel("Time (s)")
plt.legend()
plt.grid()

plt.show()


In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
import math

# =========================================================
# Parámetros
# =========================================================
NUM_ELEMENTS = 32
NUM_LOCAL_FREQ = 672
SAMPLES_PER_DATA_SET = 14976

NR_POLARIZATIONS = 2
NR_RECEIVERS = NUM_ELEMENTS // NR_POLARIZATIONS
NUM_BASELINES = NR_RECEIVERS * (NR_RECEIVERS + 1) // 2

H5_PATH = "/hdd/32_test/260319T154442Z_CHARTS_hdf5/baseband_virtual.h5"
GPU_PATH = "/home/juan_pablo/kotekan/frames_host_corr/frame_0.bin"

BEST_K = 9
BEST_REVERSE_ANT = True

FREQ_0 = 300.0
DELTA_FREQ = 0.3
frequencies = FREQ_0 + np.arange(NUM_LOCAL_FREQ) * DELTA_FREQ

ANT_REF = 16
PAIRS_TO_PLOT = [(ANT_REF, b) for b in range(NUM_ELEMENTS) if b != ANT_REF]
N_COLS = 4

# =========================================================
# Utilidades
# =========================================================
def unpack_4bit_complex(u8: np.ndarray) -> np.ndarray:
    real = (u8 >> 4).astype(np.int8)
    imag = (u8 & 0x0F).astype(np.int8)
    real[real >= 8] -= 16
    imag[imag >= 8] -= 16
    return real.astype(np.float32) + 1j * imag.astype(np.float32)

def load_gpu_corr_frame(path):
    expected_n = NUM_LOCAL_FREQ * NUM_BASELINES * NR_POLARIZATIONS * NR_POLARIZATIONS * 2
    raw = np.fromfile(path, dtype=np.int32)
    if raw.size != expected_n:
        raise ValueError(
            f"{path}: tamaño inesperado. "
            f"Leídos={raw.size}, esperados={expected_n}"
        )

    corr_i32 = raw.reshape(
        NUM_LOCAL_FREQ,
        NUM_BASELINES,
        NR_POLARIZATIONS,
        NR_POLARIZATIONS,
        2
    )
    return corr_i32[..., 0].astype(np.int64) + 1j * corr_i32[..., 1].astype(np.int64)

def load_h5_block_as_fet(h5_path, k, reverse_ant=False):
    t0 = k * SAMPLES_PER_DATA_SET
    t1 = (k + 1) * SAMPLES_PER_DATA_SET

    with h5py.File(h5_path, "r") as f:
        packed = f["baseband"][:, :, t0:t1]   # [ant, freq, time]

    if reverse_ant:
        packed = packed[::-1, :, :]

    x = unpack_4bit_complex(packed)   # [ant, freq, time]
    x = np.transpose(x, (1, 0, 2))    # [freq, ant, time]
    return np.ascontiguousarray(x)

def baseline_index(recv_y, recv_x):
    return recv_y * (recv_y + 1) // 2 + recv_x

def build_cpu_corr_in_gpu_layout_from_fet(x_fet):
    x = np.ascontiguousarray(
        x_fet.reshape(NUM_LOCAL_FREQ, NR_RECEIVERS, NR_POLARIZATIONS, x_fet.shape[2])
    )  # [f, recv, pol, t]

    cpu_corr = np.zeros(
        (NUM_LOCAL_FREQ, NUM_BASELINES, NR_POLARIZATIONS, NR_POLARIZATIONS),
        dtype=np.complex128
    )

    for recv_y in range(NR_RECEIVERS):
        Y = x[:, recv_y, :, :]
        for recv_x in range(recv_y + 1):
            X = x[:, recv_x, :, :]
            vis = np.einsum("fpt,fqt->fpq", Y, np.conj(X), optimize=True)
            b = baseline_index(recv_y, recv_x)
            cpu_corr[:, b, :, :] = vis

    return cpu_corr

# ---------------------------------------------------------
# OJO:
# Mantengo la misma convención que te funcionó antes
# para mapear "antena lógica" -> (receiver, pol)
# ---------------------------------------------------------
def elem_to_recv_pol(a):
    a_raw = NUM_ELEMENTS - 1 - a
    recv = a_raw // 2
    pol = a_raw % 2
    return recv, pol

def extract_pair_from_gpu_layout(corr_gpu_layout, a, b):
    recv_a, pol_a = elem_to_recv_pol(a)
    recv_b, pol_b = elem_to_recv_pol(b)

    if recv_b <= recv_a:
        bidx = baseline_index(recv_a, recv_b)
        return corr_gpu_layout[:, bidx, pol_a, pol_b]
    else:
        bidx = baseline_index(recv_b, recv_a)
        return np.conj(corr_gpu_layout[:, bidx, pol_b, pol_a])

# =========================================================
# Cargar mejor bloque H5 y correlación GPU
# =========================================================
gpu_corr = load_gpu_corr_frame(GPU_PATH)
x_h5 = load_h5_block_as_fet(H5_PATH, BEST_K, reverse_ant=BEST_REVERSE_ANT)
cpu_corr = build_cpu_corr_in_gpu_layout_from_fet(x_h5)

# =========================================================
# Figura 1: amplitud CPU vs GPU
# =========================================================
n_pairs = len(PAIRS_TO_PLOT)
n_rows = math.ceil(n_pairs / N_COLS)

fig, axes = plt.subplots(n_rows, N_COLS, figsize=(20, 4 * n_rows), sharex=True)
axes = np.atleast_1d(axes).ravel()

for ax, (a, b) in zip(axes, PAIRS_TO_PLOT):
    cpu_pair = extract_pair_from_gpu_layout(cpu_corr, a, b)
    gpu_pair = extract_pair_from_gpu_layout(gpu_corr, a, b)

    ax.plot(frequencies, np.abs(gpu_pair), label="GPU", linewidth=1.2)
    ax.plot(frequencies, np.abs(cpu_pair), "--", label="CPU(H5)", linewidth=1.0)

    err = np.linalg.norm(cpu_pair - gpu_pair) / np.linalg.norm(gpu_pair)
    ax.set_title(f"({a},{b})  rel_err={err:.3e}")
    ax.grid(alpha=0.3)

for ax in axes[n_pairs:]:
    ax.axis("off")

axes[0].legend()
fig.suptitle("Antena 16 vs resto — Amplitud de correlaciones", fontsize=14)
fig.supxlabel("Frecuencia [MHz]")
fig.supylabel("|Visibilidad|")
plt.tight_layout()
plt.show()

# =========================================================
# Figura 2: diferencia de fase
# dphi = angle(cpu * conj(gpu))
# enmascarando bins de muy baja magnitud
# =========================================================
fig, axes = plt.subplots(n_rows, N_COLS, figsize=(20, 4 * n_rows), sharex=True)
axes = np.atleast_1d(axes).ravel()

for ax, (a, b) in zip(axes, PAIRS_TO_PLOT):
    cpu_pair = extract_pair_from_gpu_layout(cpu_corr, a, b)
    gpu_pair = extract_pair_from_gpu_layout(gpu_corr, a, b)

    mag_ref = np.maximum(np.abs(cpu_pair), np.abs(gpu_pair))
    thr = np.quantile(mag_ref, 0.25)   # oculta el 25% más débil
    mask = mag_ref > thr

    dphi = np.full(NUM_LOCAL_FREQ, np.nan)
    dphi[mask] = np.angle(cpu_pair[mask] * np.conj(gpu_pair[mask]))

    med = np.nanmedian(np.abs(dphi))
    ax.plot(frequencies, dphi, linewidth=1.0)
    ax.axhline(0.0, linestyle="--", linewidth=0.8)

    ax.set_title(f"({a},{b})  med|Δφ|={med:.3e} rad")
    ax.set_ylim(-np.pi, np.pi)
    ax.grid(alpha=0.3)

for ax in axes[n_pairs:]:
    ax.axis("off")

fig.suptitle("Antena 16 vs resto — Diferencia de fase CPU(H5) vs GPU", fontsize=14)
fig.supxlabel("Frecuencia [MHz]")
fig.supylabel("Δφ [rad]")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# Parámetros de ploteo
# =========================================================
ANT_REF = 16
PAIRS_TO_PLOT = [(ANT_REF, b) for b in range(8, 32)]   # 8..31 inclusive

frequencies = np.linspace(300, 501.6, 672, endpoint=False)

# =========================================================
# Figura única: solo diferencia de fase
# =========================================================
n_cols = 4
n_plots = len(PAIRS_TO_PLOT)
n_rows = int(np.ceil(n_plots / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(18, 3.5 * n_rows),
    sharex=False,
    sharey=False
)
axes_flat = np.atleast_1d(axes).flatten()

for idx, (a, b) in enumerate(PAIRS_TO_PLOT):
    cpu_pair = extract_pair_from_gpu_layout(cpu_corr, a, b)
    gpu_pair = extract_pair_from_gpu_layout(gpu_corr, a, b)

    phase_diff = np.angle(cpu_pair * np.conj(gpu_pair))

    ax = axes_flat[idx]
    ax.scatter(frequencies, phase_diff, s=5)
    ax.set_title(f"Phase diff ({a},{b})", fontsize=10)
    ax.set_xlabel("Frequency [MHz]", fontsize=8)
    ax.set_ylabel("Phase diff [rad]", fontsize=8)
    ax.grid(True, alpha=0.5)

# apagar ejes sobrantes
for k in range(n_plots, len(axes_flat)):
    axes_flat[k].axis("off")

fig.suptitle(
    f"Phase difference CPU(H5)-GPU for fixed antenna {ANT_REF}",
    fontsize=16
)

plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.show()